# Example: Expectation values with FiQCIEstimator

For configuration options, mitigation levels, and result methods, see the [FiQCIEstimator guide](../pages/FiQCIEstimatorUsage.rst).

## Import everything needed

In [3]:
from fiqci.ems import FiQCIEstimator
from iqm.qiskit_iqm import IQMProvider
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp

## Initialise FiQCIEstimator

In [ ]:
url = None
quantum_computer = None

# Connect to an IQM quantum computer using the provider
if url is not None and quantum_computer is not None:
	provider = IQMProvider(url=url, quantum_computer=quantum_computer)
	backend = provider.get_backend()
else:
	# Or using a noisy simulator
	from iqm.qiskit_iqm import IQMFakeAdonis

	backend = IQMFakeAdonis()

# Initialise FiQCI estimator with mitigation level 1 (readout error mitigation)
estimator = FiQCIEstimator(backend=backend, mitigation_level=1)

# We can view the default settings enabled for mitigation level 1
estimator.mitigator_options

## Bell State expectation values with REM

In [ ]:
# Create a Bell state circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

# Transpile for backend
qc_transpiled = transpile(qc, backend=backend, optimization_level=3)

qc_transpiled.draw("mpl")

In [ ]:
# Define observables to calculate expectation values for
observables = SparsePauliOp.from_list([("ZZ", 1), ("IX", 1)])

# Map observables to the layout of the transpiled circuit
observables_device = observables.apply_layout(qc_transpiled.layout)

# Execute on FiQCI Estimator with specified observables and shots
job = estimator.run([qc_transpiled], observables=observables_device, shots=2**10)

# Retrieve mitigated expectation values
job.expectation_values()

## Access job metadata

In [ ]:
# Get all jobs
jobs = job.jobs()

# Get the first job
job0 = jobs[0]

# Access raw counts for the first job
raw_counts = job0.result().results[0].header["fiqci_ems"]["raw_counts"]

print(raw_counts)

## Manually configure mitigation options

In [ ]:
# Initialise FiQCI sampler
estimator = FiQCIEstimator(backend=backend, mitigation_level=0)

# Config rem
estimator.rem(enabled=True, calibration_shots=2**10)

# Print current mitigator options
estimator.mitigator_options